|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 3:</h2>|<h1>PagedAttention<h1>|
|<h2>Section:</h2>|<h1>Writing the kernel<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: coalesce the block-table gather<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
# find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import torch
import cudalib

This is the gather from PagedAttention, with everything else removed.

You have a table of rows in memory, an index that names the row each output
needs, and a sum over each gathered row. There is no softmax, no head, and no
block size.

Stage 07 builds the real one. This is the part that decides the speed.

In [ ]:
### run this cell: the data, and the oracle

NUM_ROWS, ROW_LEN = 200_000, 128

table = torch.randn(NUM_ROWS, ROW_LEN, device='cuda')
index = torch.randperm(NUM_ROWS, device='cuda').to(torch.int32)  # a block table
row_sums = torch.empty(NUM_ROWS, device='cuda')

oracle = table[index.long()].sum(dim=1)
useful_bytes = NUM_ROWS * ROW_LEN * 4
print(f'{useful_bytes/1e6:.0f} MB of rows to gather')

# Exercise 1: the obvious mapping, and its cost

Give each output row one thread. Let the thread walk its row. Write the
kernel, check it against the oracle, and measure the fraction of the card's
bandwidth that you reach.

In [ ]:
NAIVE = r"""
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAException.h>
#include <torch/extension.h>

// One thread per row. It walks the whole row on its own.
__global__ void gather_naive(const float* __restrict__ table,
                             const int* __restrict__ index,
                             float* __restrict__ row_sums,
                             const int num_rows, const int row_len) {

  // which row is this thread responsible for?
  const int row =
  if (row >= num_rows) return;

  // the block table says where that row actually lives
  const float* row_values =

  // walk it
  float total = 0.f;


  row_sums[row] = total;
}

void gather(torch::Tensor table, torch::Tensor index, torch::Tensor row_sums) {
  const int num_rows = index.numel(), row_len = table.size(1);
  const int threads = 256;
  gather_naive<<<   , threads, 0, at::cuda::getCurrentCUDAStream()>>>(
      table.data_ptr<float>(), index.data_ptr<int>(), row_sums.data_ptr<float>(),
      num_rows, row_len);
  C10_CUDA_KERNEL_LAUNCH_CHECK();
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) { m.def("gather", &gather); }
"""

naive = cudalib.build_source('cc_gather_naive', NAIVE)

In [ ]:
naive.gather(table, index, row_sums)
print('correct:', torch.allclose(row_sums, oracle, rtol=1e-3, atol=1e-2))

peak = cudalib.peak_bandwidth(fresh=True)
ms_naive = 
gbs_naive = 

print(f'thread per row: {ms_naive:.3f} ms  {gbs_naive:.0f} GB/s  {100*gbs_naive/peak:.0f}% of peak')

# Exercise 2: predict the fix before you write it

Do not measure first. Use the sector arithmetic. How much faster can a
coalesced version be?

You will not find a single number, and that is the interesting part. Give a
bound on each side instead. State the worst case for the naive mapping, and
the best case.

Where the measurement lands between those bounds is a fact about the cache.
You learn that fact only if you write the bounds down first.

In [ ]:
floats_per_sector = 

# At ONE instruction, how many sectors does the warp touch? How much of each
# one does it use? That gives the worst case.
worst_case = 

# Now take the NEXT instruction. The thread reads row_values[element+1]. Where does that
# byte sit? Did you pay for it already? That gives the best case.
best_case = 

print(f'stride between neighbouring threads: {ROW_LEN} floats')
print(f'speedup if nothing is cached:        {worst_case}x')
print(f'speedup if the cache catches it all: {best_case}x')

# Exercise 3: one warp per row

The arithmetic stays the same. The answer stays the same. Change only which
thread touches which byte. The 32 lanes of a warp walk one row together. They
then combine their partial sums with a shuffle.

In [ ]:
COALESCED = r"""
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAException.h>
#include <torch/extension.h>

// One WARP per row. The 32 lanes walk the row together, so at every step they
// read 32 consecutive floats: one transaction instead of 32.
__global__ void gather_coalesced(const float* __restrict__ table,
                                 const int* __restrict__ index,
                                 float* __restrict__ row_sums,
                                 const int num_rows, const int row_len) {

  // careful: the thread index now picks a LANE, not a row
  const int warp =
  const int lane =
  if (warp >= num_rows) return;

  const float* row_values = table + (long)index[warp] * row_len;

  // lane L takes elements L, L+32, L+64, ... so neighbouring lanes are
  // always on neighbouring addresses
  float total = 0.f;


  // the row's total is now spread across 32 registers. Butterfly them
  // together with __shfl_xor_sync. Every lane in the mask must reach it.


  if (lane == 0) row_sums[warp] = total;
}

void gather(torch::Tensor table, torch::Tensor index, torch::Tensor row_sums) {
  const int num_rows = index.numel(), row_len = table.size(1);
  const int threads = 256, warps_per_block = threads / 32;

  // how many blocks now? each block only covers warps_per_block rows.
  gather_coalesced<<<   , threads, 0,
                     at::cuda::getCurrentCUDAStream()>>>(
      table.data_ptr<float>(), index.data_ptr<int>(), row_sums.data_ptr<float>(),
      num_rows, row_len);
  C10_CUDA_KERNEL_LAUNCH_CHECK();
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) { m.def("gather", &gather); }
"""

fast = cudalib.build_source('cc_gather_fast', COALESCED)

In [ ]:
fast.gather(table, index, row_sums)
print('correct:', torch.allclose(row_sums, oracle, rtol=1e-3, atol=1e-2))

ms_fast = 
gbs_fast = 

print(f'thread per row: {ms_naive:.3f} ms  {gbs_naive:6.0f} GB/s  {100*gbs_naive/peak:3.0f}% of peak')
print(f'warp per row:   {ms_fast:.3f} ms  {gbs_fast:6.0f} GB/s  {100*gbs_fast/peak:3.0f}% of peak')
print(f'\nmeasured speedup:  {ms_naive/ms_fast:.2f}x')
print(f'predicted range:   {best_case}x .. {worst_case}x')

### Before you move on

Three things in your output deserve one sentence each. Write them before you
open the solution:

1. Where inside your Exercise 2 bounds did the measurement land? Near the
   worst case, or near the best?
2. A thread in the naive kernel reads `row_values[element]`, then `row_values[element+1]`. Did it pay
   for the second byte already? So what did the naive mapping waste, if not
   bytes?
3. The coalesced version probably reports more than 100 percent of peak
   bandwidth. That is not a measurement bug. What does `useful_bytes` count?
   What does the hardware move?

Then do this to the real kernel:

    ./vc guide 8b